In [3]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
import numpy as np

# 1. Load wildfire data (with latitude/longitude)
wildfire_df = pd.read_csv("../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv")

# 2. Construct proper startdate column
# Rename date columns to match what pd.to_datetime expects
wildfire_df = wildfire_df.rename(columns={
    'startdateyear': 'year',
    'startdatemonth': 'month',
    'startdateday': 'day'
})

# Construct proper datetime column
wildfire_df['startdate'] = pd.to_datetime(
    wildfire_df[['year', 'month', 'day']],
    errors='coerce'
)

# 3. Drop rows with missing coordinates
wildfire_df = wildfire_df.dropna(subset=['latitude', 'longitude'])

# 4. Set centroid lat/lon from those columns
wildfire_df['centroid_lat'] = wildfire_df['latitude']
wildfire_df['centroid_lon'] = wildfire_df['longitude']

# 5. Load airport locations
airport_df = pd.read_csv("../data/processed_data/airports_runways_joined.csv")
airport_df = airport_df.dropna(subset=['latitude_deg', 'longitude_deg'])

# 6. Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a)) * 1000  # meters

# 7. Match each wildfire to nearest airport
results = []

for _, fire in wildfire_df.iterrows():
    lat1, lon1 = fire['centroid_lat'], fire['centroid_lon']
    distances = haversine(lat1, lon1, airport_df['latitude_deg'], airport_df['longitude_deg'])
    min_idx = np.argmin(distances)
    nearest_airport = airport_df.iloc[min_idx]

    results.append({
        # Wildfire details
        'fire_id': fire['unique_id'],
        'fire_lat': lat1,
        'fire_lon': lon1,
        'startdate': fire['startdate'],
        'duration': fire['duration'],
        'size (acres)': fire['size (acres)'], 
        'fire_spread (acres/day)': fire['fire_spread (acres/day)'],  # same conversion,
        'gacc': fire['gacc'],

        # Distance
        'distance_nm': distances[min_idx] / 1852,  # in nautical miles

        # All airport details
        'ident': nearest_airport['ident'],
        'iata_code': nearest_airport['iata_code'],
        'icao_code': nearest_airport['icao_code'],
        'local_code': nearest_airport['local_code'],
        'closet_airport_name': nearest_airport['name'],
        'type': nearest_airport['type'],
        'latitude_deg': nearest_airport['latitude_deg'],
        'longitude_deg': nearest_airport['longitude_deg'],
        'elevation_ft': nearest_airport['elevation_ft'],
        'country_name': nearest_airport['country_name'],
        'region_name': nearest_airport['region_name'],
        'runway_lengths_ft': nearest_airport['runway_lengths_ft'],
        'runway_surfaces': nearest_airport['runway_surfaces'],
        'airtanker_base': nearest_airport['airtanker_base']
    })

# 6. Create DataFrame
result_df = pd.DataFrame(results)

# 7. Preview
print(result_df.head())
result_df.to_csv("../data/processed_data/nearest_airport_airtanker_bases_to_fires_final.csv", index=False)

      fire_id  fire_lat  fire_lon  startdate  duration  size (acres)  \
0  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
1  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
2  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
3  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
4  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   

   fire_spread (acres/day)                               gacc  distance_nm  \
0                 153.2547  Southern Area Coordination Center    18.608426   
1                 153.2547  Southern Area Coordination Center    18.608426   
2                 153.2547  Southern Area Coordination Center    18.608426   
3                 153.2547  Southern Area Coordination Center    18.608426   
4                 153.2547  Southern Area Coordination Center    18.608426   

  ident  ... closet_airport_name            type latitude_deg longitude_deg  \
0  KCEW  ...   Bob 

In [4]:
result_df.shape

(32082, 23)

In [3]:
import pandas as pd
import numpy as np

# 1. Load wildfire and airport data
wildfire_df = pd.read_csv("../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv")
airport_df = pd.read_csv("../data/processed_data/airports_runways_joined.csv")

# 2. Clean wildfire data
wildfire_df = wildfire_df.rename(columns={
    'startdateyear': 'year',
    'startdatemonth': 'month',
    'startdateday': 'day'
})
wildfire_df['startdate'] = pd.to_datetime(wildfire_df[['year', 'month', 'day']], errors='coerce')
wildfire_df = wildfire_df.dropna(subset=['latitude', 'longitude'])

wildfire_df['centroid_lat'] = wildfire_df['latitude']
wildfire_df['centroid_lon'] = wildfire_df['longitude']

# 3. Clean airport data
airport_df = airport_df.dropna(subset=['latitude_deg', 'longitude_deg'])

# 4. Haversine distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius (km)
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a)) * 1000  # meters

# 5. Match wildfires to nearest airport and airtanker base
results = []

for _, fire in wildfire_df.iterrows():
    lat1, lon1 = fire['centroid_lat'], fire['centroid_lon']
    
    # Distances to all airports
    all_distances = haversine(lat1, lon1, airport_df['latitude_deg'], airport_df['longitude_deg'])
    all_distances_array = all_distances.values
    
    # Nearest airport (any)
    idx_airport = np.argmin(all_distances_array)
    nearest_airport = airport_df.iloc[idx_airport]
    
    # Append airport row
    results.append({
        'fire_id': fire['unique_id'],
        'fire_lat': lat1,
        'fire_lon': lon1,
        'startdate': fire['startdate'],
        'duration': fire['duration'],
        'size (acres)': fire['size (acres)'],
        'fire_spread (acres/day)': fire['fire_spread (acres/day)'],
        'gacc': fire['gacc'],
        'distance_nm': all_distances_array[idx_airport] / 1852,
        'ident': nearest_airport['ident'],
        'iata_code': nearest_airport['iata_code'],
        'icao_code': nearest_airport['icao_code'],
        'local_code': nearest_airport['local_code'],
        'closet_airport_name': nearest_airport['name'],
        'type': nearest_airport['type'],
        'latitude_deg': nearest_airport['latitude_deg'],
        'longitude_deg': nearest_airport['longitude_deg'],
        'elevation_ft': nearest_airport['elevation_ft'],
        'country_name': nearest_airport['country_name'],
        'region_name': nearest_airport['region_name'],
        'runway_lengths_ft': nearest_airport['runway_lengths_ft'],
        'runway_surfaces': nearest_airport['runway_surfaces'],
        'airtanker_base': False
    })

    # Nearest airtanker base (only where airtanker_base == True)
    airtanker_df = airport_df[airport_df['airtanker_base'] == True]
    if not airtanker_df.empty:
        base_distances = haversine(lat1, lon1, airtanker_df['latitude_deg'], airtanker_df['longitude_deg'])
        base_distances_array = base_distances.values
        idx_base = np.argmin(base_distances_array)
        nearest_base = airtanker_df.iloc[idx_base]

        results.append({
            'fire_id': fire['unique_id'],
            'fire_lat': lat1,
            'fire_lon': lon1,
            'startdate': fire['startdate'],
            'duration': fire['duration'],
            'size (acres)': fire['size (acres)'],
            'fire_spread (acres/day)': fire['fire_spread (acres/day)'],
            'gacc': fire['gacc'],
            'distance_nm': base_distances_array[idx_base] / 1852,
            'ident': nearest_base['ident'],
            'iata_code': nearest_base['iata_code'],
            'icao_code': nearest_base['icao_code'],
            'local_code': nearest_base['local_code'],
            'closet_airport_name': nearest_base['name'],
            'type': nearest_base['type'],
            'latitude_deg': nearest_base['latitude_deg'],
            'longitude_deg': nearest_base['longitude_deg'],
            'elevation_ft': nearest_base['elevation_ft'],
            'country_name': nearest_base['country_name'],
            'region_name': nearest_base['region_name'],
            'runway_lengths_ft': nearest_base['runway_lengths_ft'],
            'runway_surfaces': nearest_base['runway_surfaces'],
            'airtanker_base': True
        })

# 6. Convert to DataFrame and save
result_df = pd.DataFrame(results)
result_df.to_csv("../data/processed_data/wildfires_with_nearest_airport_and_airtankerbase.csv", index=False)
print(result_df.head())


      fire_id  fire_lat  fire_lon  startdate  duration  size (acres)  \
0  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
1  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
2  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
3  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   
4  2020_22784     30.65    -86.85 2020-01-06         8    1218.62205   

   fire_spread (acres/day)                               gacc  distance_nm  \
0                 153.2547  Southern Area Coordination Center    18.608426   
1                 153.2547  Southern Area Coordination Center   403.525444   
2                 153.2547  Southern Area Coordination Center    18.608426   
3                 153.2547  Southern Area Coordination Center   403.525444   
4                 153.2547  Southern Area Coordination Center    18.608426   

  ident  ... closet_airport_name            type latitude_deg longitude_deg  \
0  KCEW  ...   Bob 